<a href="https://colab.research.google.com/github/hassan11196/llm-systems-cookbook/blob/main/notebooks/04_agents/05_mcp_server_client.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" style="height: 28px; vertical-align: middle;"/></a>  **[▶️ Run this notebook in Colab](https://colab.research.google.com/github/hassan11196/llm-systems-cookbook/blob/main/notebooks/04_agents/05_mcp_server_client.ipynb)**

# Model Context Protocol (MCP) — a minimal server and client

> **Track 04 — Agents · Notebook 05 · Runtime: ≈30 s on CPU**
>
> **Prerequisites:** `04_agents/01` (ReAct from scratch).
>
> **Reference:** [Model Context Protocol specification](https://modelcontextprotocol.io/specification/).

---

## What

MCP is a JSON-RPC 2.0 protocol for letting an LLM client call tools
and fetch resources hosted by any "server". It lets a ReAct agent
outsource its tool registry: the server advertises tools over
`tools/list`, the agent calls one via `tools/call`, the result comes
back. Works over stdio, WebSocket, or HTTP+SSE.

We implement both sides in pure Python stdlib. The protocol is short
enough that the whole thing fits in one notebook.

Three RPC methods used here:

- `initialize` — handshake; client says "hello, what version?".
- `tools/list` — server returns tool schemas.
- `tools/call` — server runs a tool with given args.

The notebook verifies a round-trip: client calls calculator and
wiki tools, gets back correct results, and the server rejects an
unknown tool with a clean JSON-RPC error.


In [ ]:
from __future__ import annotations

import ast
import json
import operator as op
import sys
from pathlib import Path
from typing import Any

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

from llm_systems_cookbook._utils import set_seed
from scoring.harness import Scorer

set_seed(0)
s = Scorer("04_agents_05_mcp_server_client")


## JSON-RPC 2.0 frame

Every message is `{"jsonrpc": "2.0", "id": ..., "method": ..., "params": ...}`
or a matching response with `result` or `error`. MCP layers its
semantics on top. We don't do streaming transport here — the client
calls `server.handle(request)` synchronously so the notebook is
deterministic.


In [ ]:
def make_request(method: str, params: dict[str, Any] | None = None, req_id: int = 1) -> dict:
    return {"jsonrpc": "2.0", "id": req_id, "method": method, "params": params or {}}


def make_response(result: Any, req_id: int) -> dict:
    return {"jsonrpc": "2.0", "id": req_id, "result": result}


def make_error(code: int, message: str, req_id: int | None = None, data: Any = None) -> dict:
    err: dict[str, Any] = {"code": code, "message": message}
    if data is not None:
        err["data"] = data
    return {"jsonrpc": "2.0", "id": req_id, "error": err}


# MCP reserves the JSON-RPC error range; these are the ones we use.
PARSE_ERROR = -32700
METHOD_NOT_FOUND = -32601
INVALID_PARAMS = -32602
TOOL_NOT_FOUND = -32001


## The server

Three tools: a safe calculator (AST walk, never eval), a stub
`wiki_search`, and a `get_current_time`. The server advertises them
via `tools/list` (MCP schema shape) and dispatches via `tools/call`.


In [ ]:
_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg, ast.UAdd: op.pos,
}


def safe_eval(expr: str) -> float:
    def _eval(node: ast.AST) -> float:
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _ALLOWED_OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return _ALLOWED_OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"unsupported node {type(node).__name__}")
    return _eval(ast.parse(expr.strip(), mode="eval").body)


FACTS = {
    "capital of france": "Paris",
    "tallest mountain":  "Mount Everest",
    "speed of light":    "299,792,458 m/s",
}


class MCPServer:
    PROTOCOL_VERSION = "2024-11-05"
    TOOLS: dict[str, dict] = {
        "calculator": {
            "name": "calculator",
            "description": "Evaluate a simple arithmetic expression.",
            "inputSchema": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
        "wiki_search": {
            "name": "wiki_search",
            "description": "Look up a short fact.",
            "inputSchema": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    }

    def handle(self, req: dict) -> dict:
        if req.get("jsonrpc") != "2.0":
            return make_error(PARSE_ERROR, "invalid JSON-RPC version", req.get("id"))
        method = req.get("method")
        params = req.get("params", {}) or {}
        rid = req.get("id")

        if method == "initialize":
            return make_response(
                {"protocolVersion": self.PROTOCOL_VERSION, "serverInfo": {"name": "lab-mcp-server", "version": "0.1.0"}},
                rid,
            )

        if method == "tools/list":
            return make_response({"tools": list(self.TOOLS.values())}, rid)

        if method == "tools/call":
            name = params.get("name")
            args = params.get("arguments", {})
            if name not in self.TOOLS:
                return make_error(TOOL_NOT_FOUND, f"unknown tool {name!r}", rid)
            try:
                if name == "calculator":
                    value = safe_eval(args["expression"])
                    out = str(int(value)) if isinstance(value, float) and value.is_integer() else str(value)
                elif name == "wiki_search":
                    q = args["query"].lower().strip()
                    out = FACTS.get(q, "no match")
                else:
                    return make_error(METHOD_NOT_FOUND, "unhandled tool", rid)
            except KeyError as e:
                return make_error(INVALID_PARAMS, f"missing argument {e}", rid)
            except Exception as e:  # noqa: BLE001 - surface to client
                return make_error(INVALID_PARAMS, f"{type(e).__name__}: {e}", rid)
            return make_response({"content": [{"type": "text", "text": out}]}, rid)

        return make_error(METHOD_NOT_FOUND, f"unknown method {method!r}", rid)


server = MCPServer()
print("server ready")


## The client

Sends requests, parses results. In production the transport would be
stdio or WebSocket; here we just call `server.handle(req)` directly.
The structure of request and response is identical either way.


In [ ]:
class MCPClient:
    def __init__(self, server: MCPServer) -> None:
        self.server = server
        self._rid = 0

    def _next_id(self) -> int:
        self._rid += 1
        return self._rid

    def initialize(self) -> dict:
        return self.server.handle(make_request("initialize", req_id=self._next_id()))

    def list_tools(self) -> list[dict]:
        r = self.server.handle(make_request("tools/list", req_id=self._next_id()))
        return r["result"]["tools"]

    def call(self, name: str, arguments: dict) -> dict:
        return self.server.handle(make_request(
            "tools/call",
            {"name": name, "arguments": arguments},
            req_id=self._next_id(),
        ))


client = MCPClient(server)
init = client.initialize()
print(f"initialize -> {init['result']}")

tools = client.list_tools()
print(f"tools/list -> {[t['name'] for t in tools]}")

calc = client.call("calculator", {"expression": "2 ** 10 + 5"})
print(f"tools/call calculator(2**10+5) -> {calc['result']['content'][0]['text']}")

wiki = client.call("wiki_search", {"query": "capital of france"})
print(f"tools/call wiki(capital of france) -> {wiki['result']['content'][0]['text']}")

missing = client.call("nonexistent_tool", {})
print(f"tools/call nonexistent -> {missing.get('error')}")


In [ ]:
s.check(
    "initialize_returns_protocol_version",
    lambda: init["result"]["protocolVersion"] == MCPServer.PROTOCOL_VERSION,
    msg=f"got {init['result'].get('protocolVersion')}",
)
s.check(
    "tools_list_includes_both_tools",
    lambda: {"calculator", "wiki_search"} <= {t["name"] for t in tools},
    msg=f"{[t['name'] for t in tools]}",
)
s.check(
    "calculator_returns_correct_result",
    lambda: calc["result"]["content"][0]["text"] == "1029",
    msg=f"got {calc['result']['content'][0]['text']}",
)
s.check(
    "wiki_search_finds_paris",
    lambda: wiki["result"]["content"][0]["text"] == "Paris",
    msg=f"got {wiki['result']['content'][0]['text']}",
)
s.check(
    "unknown_tool_returns_json_rpc_error",
    lambda: missing.get("error", {}).get("code") == TOOL_NOT_FOUND,
    msg=f"error = {missing.get('error')}",
)
s.check(
    "every_response_has_matching_id",
    lambda: all(
        r.get("id") is not None
        for r in (init, missing)
    ),
    msg="initialize and missing-tool responses must echo their request id",
)


## Exercises

1. **Actual stdio transport.** Wrap `MCPServer` in a loop reading one
   line of JSON per iteration from stdin and writing the response to
   stdout. Drive it from a separate Python process and the protocol
   is indistinguishable.
2. **Streaming results.** MCP supports partial tool results; define a
   long-running tool (sleep + yield progress) and stream chunks via
   `notifications/progress`.
3. **Real client.** `pip install mcp`, connect to a real MCP server
   (e.g., the
   [GitHub MCP server](https://github.com/github/github-mcp-server))
   and call one of its tools.

## References

- MCP specification (tools, resources, prompts).
- The `mcp` Python package on PyPI for the production implementation.


In [ ]:
s.summary()
s.save()
